<a href="https://colab.research.google.com/github/bojannithya-tech/nithyabojan/blob/main/AI_POS_Transaction_Anomaly_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Part 1: Synthetic POS Transaction Dataset

Part 1A — Create the first 3 POS transactions

In [1]:
# Synthetic POS transaction dataset
# We start with 3 scenarios:
# 1. Normal cashback
# 2. Higher-than-expected cashback
# 3. Negative cashback

transactions = [
    {
        "transaction_id": "TXN001",
        "store_id": "STORE101",
        "lane_id": "LANE01",

        "items": [
            {"item_id": "ITEM001", "price": 20.00, "quantity": 2},
            {"item_id": "ITEM002", "price": 10.00, "quantity": 1}
        ],

        "basket_total": 50.00,

        "promotion": {
            "discount_amount": 5.00
        },

        "gift_card": {
            "used": False,
            "amount": 0.00,
            "activation_status": "NOT_APPLICABLE",
            "failure_amount": 0.00
        },

        "payment": {
            "type": "DEBIT",
            "amount": 45.00,
            "status": "SUCCESS"
        },

        "cashback": {
            "expected": 5.00,
            "actual": 5.00
        }
    },

    {
        "transaction_id": "TXN002",
        "store_id": "STORE101",
        "lane_id": "LANE02",

        "items": [
            {"item_id": "ITEM101", "price": 40.00, "quantity": 1},
            {"item_id": "ITEM102", "price": 30.00, "quantity": 1}
        ],

        "basket_total": 70.00,

        "promotion": {
            "discount_amount": 5.00
        },

        "gift_card": {
            "used": True,
            "amount": 25.00,
            "activation_status": "FAILED",
            "failure_amount": 25.00
        },

        "payment": {
            "type": "CREDIT",
            "amount": 70.00,
            "status": "SUCCESS"
        },

        "cashback": {
            "expected": 5.00,
            "actual": 30.00
        }
    },

    {
        "transaction_id": "TXN003",
        "store_id": "STORE102",
        "lane_id": "LANE03",

        "items": [
            {"item_id": "ITEM201", "price": 50.00, "quantity": 1},
            {"item_id": "ITEM202", "price": 20.00, "quantity": 1}
        ],

        "basket_total": 70.00,

        "promotion": {
            "discount_amount": 10.00
        },

        "gift_card": {
            "used": False,
            "amount": 0.00,
            "activation_status": "NOT_APPLICABLE",
            "failure_amount": 0.00
        },

        "payment": {
            "type": "DEBIT",
            "amount": 60.00,
            "status": "SUCCESS"
        },

        "cashback": {
            "expected": 5.00,
            "actual": -15.00
        }
    }
]

print("Synthetic transactions created:", len(transactions))

Synthetic transactions created: 3


Part 1B — Display the transactions

In [2]:
import json

for transaction in transactions:
    print("=" * 60)
    print(json.dumps(transaction, indent=2))

{
  "transaction_id": "TXN001",
  "store_id": "STORE101",
  "lane_id": "LANE01",
  "items": [
    {
      "item_id": "ITEM001",
      "price": 20.0,
      "quantity": 2
    },
    {
      "item_id": "ITEM002",
      "price": 10.0,
      "quantity": 1
    }
  ],
  "basket_total": 50.0,
  "promotion": {
    "discount_amount": 5.0
  },
  "gift_card": {
    "used": false,
    "amount": 0.0,
    "activation_status": "NOT_APPLICABLE",
    "failure_amount": 0.0
  },
  "payment": {
    "type": "DEBIT",
    "amount": 45.0,
    "status": "SUCCESS"
  },
  "cashback": {
    "expected": 5.0,
    "actual": 5.0
  }
}
{
  "transaction_id": "TXN002",
  "store_id": "STORE101",
  "lane_id": "LANE02",
  "items": [
    {
      "item_id": "ITEM101",
      "price": 40.0,
      "quantity": 1
    },
    {
      "item_id": "ITEM102",
      "price": 30.0,
      "quantity": 1
    }
  ],
  "basket_total": 70.0,
  "promotion": {
    "discount_amount": 5.0
  },
  "gift_card": {
    "used": true,
    "amount": 25.0,


Part 2 — Build the Transaction Health Check

## Part 2: Transaction Health Check

The Transaction Health Check performs deterministic validation
before invoking GenAI agents.

It classifies a completed POS transaction as:

- NORMAL
- HIGH_CASHBACK
- NEGATIVE_CASHBACK

Financial calculations are performed using deterministic Python
logic rather than relying on an LLM.

In [3]:
def check_transaction_health(transaction):

    expected = transaction["cashback"]["expected"]
    actual = transaction["cashback"]["actual"]

    variance = round(actual - expected, 2)

    # Negative cashback has highest priority
    if actual < 0:
        status = "ANOMALY"
        anomaly_type = "NEGATIVE_CASHBACK"

    # Cashback greater than expected
    elif actual > expected:
        status = "ANOMALY"
        anomaly_type = "HIGH_CASHBACK"

    else:
        status = "NORMAL"
        anomaly_type = "NONE"

    return {
        "transaction_id": transaction["transaction_id"],
        "status": status,
        "anomaly_type": anomaly_type,
        "expected_cashback": expected,
        "actual_cashback": actual,
        "variance": variance
    }

Part 2B — Test the Health Check

In [4]:
health_results = []

for transaction in transactions:

    result = check_transaction_health(transaction)
    health_results.append(result)

    print("=" * 50)
    print("Transaction ID :", result["transaction_id"])
    print("Status         :", result["status"])
    print("Anomaly Type   :", result["anomaly_type"])
    print("Expected       : $", result["expected_cashback"])
    print("Actual         : $", result["actual_cashback"])
    print("Variance       : $", result["variance"])

Transaction ID : TXN001
Status         : NORMAL
Anomaly Type   : NONE
Expected       : $ 5.0
Actual         : $ 5.0
Variance       : $ 0.0
Transaction ID : TXN002
Status         : ANOMALY
Anomaly Type   : HIGH_CASHBACK
Expected       : $ 5.0
Actual         : $ 30.0
Variance       : $ 25.0
Transaction ID : TXN003
Status         : ANOMALY
Anomaly Type   : NEGATIVE_CASHBACK
Expected       : $ 5.0
Actual         : $ -15.0
Variance       : $ -20.0


Part 3 — Pattern & Correlation Analysis

## Part 3: Pattern and Correlation Analysis

For anomalous transactions, the system investigates whether the
cashback variance correlates with other monetary values in the
transaction.

The analysis checks relationships with:

- Gift card amount
- Gift card failure amount
- Promotion/discount amount
- Payment amount
- Individual item amounts

The detected correlations are treated as investigation evidence,
not as confirmed root causes.

Create the correlation tool

In [5]:
def analyze_amount_correlations(transaction, health_result):

    # Normal transactions don't require deeper investigation
    if health_result["status"] == "NORMAL":
        return {
            "transaction_id": transaction["transaction_id"],
            "correlations": [],
            "message": "No anomaly detected. Correlation analysis not required."
        }

    expected = health_result["expected_cashback"]
    actual = health_result["actual_cashback"]

    # For HIGH cashback, investigate the excess.
    # For NEGATIVE cashback, investigate the magnitude of the negative value.
    if health_result["anomaly_type"] == "HIGH_CASHBACK":
        suspicious_amount = round(actual - expected, 2)
        investigation_basis = "EXCESS_CASHBACK"

    elif health_result["anomaly_type"] == "NEGATIVE_CASHBACK":
        suspicious_amount = round(abs(actual), 2)
        investigation_basis = "NEGATIVE_CASHBACK_MAGNITUDE"

    else:
        suspicious_amount = 0
        investigation_basis = "NONE"

    correlations = []

    # Candidate transaction values
    candidates = {
        "gift_card_amount": transaction["gift_card"]["amount"],
        "gift_card_failure_amount": transaction["gift_card"]["failure_amount"],
        "promotion_discount": transaction["promotion"]["discount_amount"],
        "payment_amount": transaction["payment"]["amount"]
    }

    # Compare suspicious amount with transaction-level values
    for name, value in candidates.items():
        if value > 0 and abs(suspicious_amount - value) < 0.01:
            correlations.append({
                "type": "EXACT_MATCH",
                "field": name,
                "value": value
            })

    # Compare against individual item amounts
    for item in transaction["items"]:

        item_total = round(item["price"] * item["quantity"], 2)

        if abs(suspicious_amount - item_total) < 0.01:
            correlations.append({
                "type": "EXACT_MATCH",
                "field": f'item_total_{item["item_id"]}',
                "value": item_total
            })

    return {
        "transaction_id": transaction["transaction_id"],
        "anomaly_type": health_result["anomaly_type"],
        "investigation_basis": investigation_basis,
        "suspicious_amount": suspicious_amount,
        "correlations": correlations
    }

Run correlation analysis

In [6]:
correlation_results = []

for transaction, health_result in zip(transactions, health_results):

    result = analyze_amount_correlations(
        transaction,
        health_result
    )

    correlation_results.append(result)

    print("=" * 60)
    print("Transaction:", transaction["transaction_id"])

    if health_result["status"] == "NORMAL":
        print("No anomaly - investigation skipped.")
        continue

    print("Anomaly:", result["anomaly_type"])
    print("Investigation Basis:", result["investigation_basis"])
    print("Suspicious Amount: $", result["suspicious_amount"])

    if result["correlations"]:
        print("\nCorrelations found:")

        for correlation in result["correlations"]:
            print(
                "  ->",
                correlation["field"],
                "= $",
                correlation["value"],
                "|",
                correlation["type"]
            )
    else:
        print("\nNo direct amount correlation found.")

Transaction: TXN001
No anomaly - investigation skipped.
Transaction: TXN002
Anomaly: HIGH_CASHBACK
Investigation Basis: EXCESS_CASHBACK
Suspicious Amount: $ 25.0

Correlations found:
  -> gift_card_amount = $ 25.0 | EXACT_MATCH
  -> gift_card_failure_amount = $ 25.0 | EXACT_MATCH
Transaction: TXN003
Anomaly: NEGATIVE_CASHBACK
Investigation Basis: NEGATIVE_CASHBACK_MAGNITUDE
Suspicious Amount: $ 15.0

No direct amount correlation found.


Add correlation strength

In [10]:
def calculate_evidence_strength(correlation_result):

    correlations = correlation_result.get("correlations", [])

    if len(correlations) >= 2:
        return "STRONG"

    elif len(correlations) == 1:
        return "MODERATE"

    else:
        return "INSUFFICIENT"

In [11]:
for result in correlation_results:

    if "correlations" not in result:
        continue

    strength = calculate_evidence_strength(result)

    result["evidence_strength"] = strength

    print(
        result["transaction_id"],
        "-> Evidence Strength:",
        strength
    )

TXN001 -> Evidence Strength: INSUFFICIENT
TXN002 -> Evidence Strength: STRONG
TXN003 -> Evidence Strength: INSUFFICIENT


Part 4 — Historical Transaction Context Analysis

## Part 4: Historical Transaction Context Analysis

Some rare POS anomalies may depend on transaction state carried across
transaction boundaries.

The Historical Context Analyzer examines previous transactions from the
same lane and checks whether earlier failure-related amounts correlate
with the current cashback anomaly.

This analysis generates investigation evidence only. A detected
correlation does not by itself prove causation.

Add a historical edge-case sequence

In [12]:
historical_transactions = [
    {
        "transaction_id": "TXN004",
        "store_id": "STORE103",
        "lane_id": "LANE05",
        "sequence": 1,

        "items": [
            {"item_id": "ITEM301", "price": 30.00, "quantity": 1}
        ],

        "basket_total": 30.00,

        "promotion": {
            "discount_amount": 0.00
        },

        "gift_card": {
            "used": True,
            "amount": 10.00,
            "activation_status": "FAILED",
            "failure_amount": 10.00
        },

        "payment": {
            "type": "CREDIT",
            "amount": 30.00,
            "status": "SUCCESS"
        },

        "cashback": {
            "expected": 0.00,
            "actual": 0.00
        }
    },

    {
        "transaction_id": "TXN005",
        "store_id": "STORE103",
        "lane_id": "LANE05",
        "sequence": 2,

        "items": [
            {"item_id": "ITEM302", "price": 40.00, "quantity": 1}
        ],

        "basket_total": 40.00,

        "promotion": {
            "discount_amount": 0.00
        },

        "gift_card": {
            "used": True,
            "amount": 15.00,
            "activation_status": "FAILED",
            "failure_amount": 15.00
        },

        "payment": {
            "type": "DEBIT",
            "amount": 40.00,
            "status": "SUCCESS"
        },

        "cashback": {
            "expected": 0.00,
            "actual": 0.00
        }
    },

    {
        "transaction_id": "TXN006",
        "store_id": "STORE103",
        "lane_id": "LANE05",
        "sequence": 3,

        "items": [
            {"item_id": "ITEM303", "price": 50.00, "quantity": 1}
        ],

        "basket_total": 50.00,

        "promotion": {
            "discount_amount": 5.00
        },

        "gift_card": {
            "used": False,
            "amount": 0.00,
            "activation_status": "NOT_APPLICABLE",
            "failure_amount": 0.00
        },

        "payment": {
            "type": "CREDIT",
            "amount": 45.00,
            "status": "SUCCESS"
        },

        "cashback": {
            "expected": 5.00,
            "actual": 30.00
        }
    }
]

print("Historical transactions created:", len(historical_transactions))

Historical transactions created: 3


4B — Historical Context Analyzer

In [13]:
def analyze_historical_context(current_transaction, transaction_history):

    current_health = check_transaction_health(current_transaction)

    # Historical analysis is only required for anomalies
    if current_health["status"] == "NORMAL":
        return {
            "transaction_id": current_transaction["transaction_id"],
            "historical_correlation": False,
            "message": "Normal transaction - historical analysis not required."
        }

    # Determine suspicious amount
    if current_health["anomaly_type"] == "HIGH_CASHBACK":
        suspicious_amount = round(
            current_health["actual_cashback"]
            - current_health["expected_cashback"],
            2
        )

    elif current_health["anomaly_type"] == "NEGATIVE_CASHBACK":
        suspicious_amount = round(
            abs(current_health["actual_cashback"]),
            2
        )

    else:
        suspicious_amount = 0.0

    # Only compare previous transactions from same store/lane
    relevant_history = [
        tx for tx in transaction_history
        if tx["store_id"] == current_transaction["store_id"]
        and tx["lane_id"] == current_transaction["lane_id"]
        and tx.get("sequence", 0) < current_transaction.get("sequence", 0)
    ]

    previous_failure_amounts = [
        tx["gift_card"]["failure_amount"]
        for tx in relevant_history
        if tx["gift_card"]["failure_amount"] > 0
    ]

    accumulated_failure_amount = round(
        sum(previous_failure_amounts),
        2
    )

    historical_match = (
        accumulated_failure_amount > 0
        and abs(suspicious_amount - accumulated_failure_amount) < 0.01
    )

    return {
        "transaction_id": current_transaction["transaction_id"],
        "anomaly_type": current_health["anomaly_type"],
        "suspicious_amount": suspicious_amount,
        "previous_failure_amounts": previous_failure_amounts,
        "accumulated_failure_amount": accumulated_failure_amount,
        "historical_correlation": historical_match
    }

Investigate TXN006

In [14]:
current_transaction = historical_transactions[2]

history_result = analyze_historical_context(
    current_transaction,
    historical_transactions
)

print("Transaction:", history_result["transaction_id"])
print("Anomaly Type:", history_result["anomaly_type"])
print("Suspicious Cashback Amount: $", history_result["suspicious_amount"])

print(
    "Previous Gift Card Failure Amounts:",
    history_result["previous_failure_amounts"]
)

print(
    "Accumulated Previous Failure Amount: $",
    history_result["accumulated_failure_amount"]
)

print(
    "Historical Correlation:",
    history_result["historical_correlation"]
)

Transaction: TXN006
Anomaly Type: HIGH_CASHBACK
Suspicious Cashback Amount: $ 25.0
Previous Gift Card Failure Amounts: [10.0, 15.0]
Accumulated Previous Failure Amount: $ 25.0
Historical Correlation: True


Generate structured evidence

In [15]:
def build_historical_evidence(history_result):

    if history_result.get("historical_correlation"):

        return {
            "evidence_type": "CROSS_TRANSACTION_CORRELATION",
            "strength": "STRONG",
            "observation": (
                f"The suspicious cashback amount of "
                f"${history_result['suspicious_amount']:.2f} "
                f"matches the accumulated previous gift-card "
                f"failure amount of "
                f"${history_result['accumulated_failure_amount']:.2f}."
            ),
            "interpretation": (
                "This may indicate that failure-related state "
                "persisted across transaction boundaries. "
                "Further investigation is required."
            )
        }

    return {
        "evidence_type": "NO_HISTORICAL_CORRELATION",
        "strength": "INSUFFICIENT",
        "observation": "No matching historical amount pattern was identified.",
        "interpretation": "No historical root-cause hypothesis can be supported."
    }


historical_evidence = build_historical_evidence(history_result)

print(json.dumps(historical_evidence, indent=2))

{
  "evidence_type": "CROSS_TRANSACTION_CORRELATION",
  "strength": "STRONG",
  "observation": "The suspicious cashback amount of $25.00 matches the accumulated previous gift-card failure amount of $25.00.",
  "interpretation": "This may indicate that failure-related state persisted across transaction boundaries. Further investigation is required."
}


Part 5 — RAG Knowledge Base

## Part 5: RAG-Based POS Knowledge Retrieval

The investigation assistant uses Retrieval-Augmented Generation (RAG)
to retrieve relevant POS business and troubleshooting knowledge.

The knowledge base contains synthetic guidance related to:

- Cashback processing
- Gift card failure handling
- Transaction state management
- Negative cashback investigation
- Escalation procedures

RAG helps ground the investigation in retrieved evidence rather than
allowing the LLM to generate unsupported explanations.

Create the synthetic knowledge base

In [17]:
knowledge_documents = [
    {
        "doc_id": "KB001",
        "title": "POS Cashback Processing Guide",
        "content": """
Cashback values must be validated against the expected cashback
calculated for the current transaction.

If actual cashback is greater than expected cashback, the transaction
must be treated as a high-cashback anomaly.

Investigation should compare the cashback variance with other monetary
values in the transaction, including promotion amounts, tender values,
gift card amounts and failure-related amounts.

An amount correlation is investigation evidence and must not by itself
be considered proof of root cause.
"""
    },

    {
        "doc_id": "KB002",
        "title": "Gift Card Failure Handling Guide",
        "content": """
Gift card processing may contain transaction-scoped values representing
failed or incomplete gift card operations.

Failure-related values should be associated with the transaction in
which the failure occurred.

When investigating abnormal cashback after a gift card failure,
engineers should verify the failure amount, activation status and
subsequent transaction processing.

Failure-related state should not incorrectly influence unrelated
transactions.
"""
    },

    {
        "doc_id": "KB003",
        "title": "POS Transaction State Management Guide",
        "content": """
Transaction-specific temporary state should be initialized correctly
for each new POS transaction.

State associated with a completed or failed transaction should not
unexpectedly affect subsequent transactions.

If an anomalous monetary value matches accumulated values from previous
transactions, investigate whether transaction-scoped state was retained
across transaction boundaries.

A cross-transaction monetary correlation should be treated as a
root-cause hypothesis requiring engineering validation.
"""
    },

    {
        "doc_id": "KB004",
        "title": "Negative Cashback Investigation Guide",
        "content": """
A negative cashback value should be treated as an anomaly and requires
investigation.

The investigation should examine adjustments, reversals, promotions,
refund-related values and tender events that may correlate with the
magnitude of the negative cashback.

If no supporting transaction evidence is available, the system should
not infer a root cause and should recommend manual investigation.
"""
    },

    {
        "doc_id": "KB005",
        "title": "POS Anomaly Escalation Guide",
        "content": """
An automated investigation should distinguish between observations,
correlations and confirmed root causes.

When evidence is insufficient or conflicting, the investigation result
should be marked as inconclusive.

Inconclusive anomalies should be escalated for manual engineering
investigation.

AI-generated hypotheses must remain subject to human review before
being accepted as a root-cause conclusion.
"""
    }
]

print("Knowledge documents created:", len(knowledge_documents))

Knowledge documents created: 5


Install embedding/vector-search libraries

In [18]:
!pip -q install sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 72.1 MB/s eta 0:00:00


Load the embedding model

In [19]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.


Create document embeddings

In [20]:
document_texts = [
    doc["title"] + "\n" + doc["content"]
    for doc in knowledge_documents
]

document_embeddings = embedding_model.encode(
    document_texts,
    convert_to_numpy=True
)

print("Number of documents:", len(document_texts))
print("Embedding shape:", document_embeddings.shape)

Number of documents: 5
Embedding shape: (5, 384)


Build the FAISS vector database

In [21]:
dimension = document_embeddings.shape[1]

faiss_index = faiss.IndexFlatL2(dimension)

faiss_index.add(
    document_embeddings.astype("float32")
)

print("Documents stored in FAISS:", faiss_index.ntotal)

Documents stored in FAISS: 5


Build the Retriever

In [22]:
def retrieve_pos_knowledge(query, top_k=2):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    distances, indices = faiss_index.search(
        query_embedding,
        top_k
    )

    results = []

    for distance, index in zip(distances[0], indices[0]):

        document = knowledge_documents[index]

        results.append({
            "doc_id": document["doc_id"],
            "title": document["title"],
            "content": document["content"].strip(),
            "distance": float(distance)
        })

    return results

Test RAG retrieval

In [23]:
query = """
A high cashback transaction has an excess amount that matches
accumulated gift card failure amounts from previous transactions.
What should be investigated?
"""

retrieved_documents = retrieve_pos_knowledge(
    query,
    top_k=2
)

for doc in retrieved_documents:

    print("=" * 70)
    print("Document:", doc["title"])
    print("Distance:", round(doc["distance"], 4))
    print()
    print(doc["content"])

Document: Gift Card Failure Handling Guide
Distance: 0.6626

Gift card processing may contain transaction-scoped values representing
failed or incomplete gift card operations.

Failure-related values should be associated with the transaction in
which the failure occurred.

When investigating abnormal cashback after a gift card failure,
engineers should verify the failure amount, activation status and
subsequent transaction processing.

Failure-related state should not incorrectly influence unrelated
transactions.
Document: POS Cashback Processing Guide
Distance: 0.8364

Cashback values must be validated against the expected cashback
calculated for the current transaction.

If actual cashback is greater than expected cashback, the transaction
must be treated as a high-cashback anomaly.

Investigation should compare the cashback variance with other monetary
values in the transaction, including promotion amounts, tender values,
gift card amounts and failure-related amounts.

An amount cor

Connect the anomaly evidence to RAG

In [24]:
def build_rag_query(health_result, historical_evidence):

    query = f"""
POS transaction anomaly investigation.

Anomaly type:
{health_result['anomaly_type']}

Expected cashback:
${health_result['expected_cashback']:.2f}

Actual cashback:
${health_result['actual_cashback']:.2f}

Cashback variance:
${health_result['variance']:.2f}

Historical evidence:
{historical_evidence['observation']}

Evidence interpretation:
{historical_evidence['interpretation']}

Retrieve POS knowledge that can help investigate this anomaly.
"""

    return query

In [25]:
txn006_health = check_transaction_health(
    historical_transactions[2]
)

rag_query = build_rag_query(
    txn006_health,
    historical_evidence
)

print(rag_query)


POS transaction anomaly investigation.

Anomaly type:
HIGH_CASHBACK

Expected cashback:
$5.00

Actual cashback:
$30.00

Cashback variance:
$25.00

Historical evidence:
The suspicious cashback amount of $25.00 matches the accumulated previous gift-card failure amount of $25.00.

Evidence interpretation:
This may indicate that failure-related state persisted across transaction boundaries. Further investigation is required.

Retrieve POS knowledge that can help investigate this anomaly.



In [26]:
txn006_knowledge = retrieve_pos_knowledge(
    rag_query,
    top_k=2
)

for doc in txn006_knowledge:

    print("=" * 70)
    print(doc["title"])
    print("=" * 70)
    print(doc["content"])

POS Cashback Processing Guide
Cashback values must be validated against the expected cashback
calculated for the current transaction.

If actual cashback is greater than expected cashback, the transaction
must be treated as a high-cashback anomaly.

Investigation should compare the cashback variance with other monetary
values in the transaction, including promotion amounts, tender values,
gift card amounts and failure-related amounts.

An amount correlation is investigation evidence and must not by itself
be considered proof of root cause.
POS Transaction State Management Guide
Transaction-specific temporary state should be initialized correctly
for each new POS transaction.

State associated with a completed or failed transaction should not
unexpectedly affect subsequent transactions.

If an anomalous monetary value matches accumulated values from previous
transactions, investigate whether transaction-scoped state was retained
across transaction boundaries.

A cross-transaction moneta

Part 6 — GenAI Investigation Agent

## Part 6: GenAI Investigation Agent

The GenAI Investigation Agent combines:

- Transaction details
- Deterministic health-check results
- Current transaction correlations
- Historical transaction evidence
- Retrieved RAG knowledge

The agent generates an evidence-grounded investigation hypothesis and
recommended next steps.

The LLM does not perform the financial calculations and must not claim
a confirmed root cause when supporting evidence is insufficient.

Install Gemini SDK

In [61]:
!pip -q install google-genai

In [62]:
from google.colab import userdata
from google import genai

api_key = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)

print("Gemini client initialized successfully.")

Gemini client initialized successfully.


In [63]:
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY").strip()

print("Key loaded:", bool(api_key))
print("Contains newline:", "\n" in api_key)
print("Starts with AQ:", api_key.startswith("AQ"))

Key loaded: True
Contains newline: False
Starts with AQ: True


In [64]:
from google import genai

client = genai.Client(api_key=api_key)

print("Client created")

Client created


In [65]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Reply with only OK"
)

print(response.text)

ClientError: 401 UNAUTHENTICATED. {'error': {'code': 401, 'message': 'Request had invalid authentication credentials. Expected OAuth 2 access token, login cookie or other valid authentication credential. See https://developers.google.com/identity/sign-in/web/devconsole-project.', 'status': 'UNAUTHENTICATED', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'ACCESS_TOKEN_TYPE_UNSUPPORTED', 'metadata': {'method': 'google.ai.generativelanguage.v1beta.GenerativeService.GenerateContent', 'service': 'generativelanguage.googleapis.com'}}]}}

Prepare the RAG context

In [53]:
def build_rag_context(retrieved_documents):

    context_parts = []

    for doc in retrieved_documents:
        context_parts.append(
            f"""
Document: {doc['title']}

{doc['content']}
"""
        )

    return "\n".join(context_parts)


rag_context = build_rag_context(txn006_knowledge)

print(rag_context)


Document: POS Cashback Processing Guide

Cashback values must be validated against the expected cashback
calculated for the current transaction.

If actual cashback is greater than expected cashback, the transaction
must be treated as a high-cashback anomaly.

Investigation should compare the cashback variance with other monetary
values in the transaction, including promotion amounts, tender values,
gift card amounts and failure-related amounts.

An amount correlation is investigation evidence and must not by itself
be considered proof of root cause.


Document: POS Transaction State Management Guide

Transaction-specific temporary state should be initialized correctly
for each new POS transaction.

State associated with a completed or failed transaction should not
unexpectedly affect subsequent transactions.

If an anomalous monetary value matches accumulated values from previous
transactions, investigate whether transaction-scoped state was retained
across transaction boundaries.

A

Build a grounded investigation prompt

In [54]:
def build_investigation_prompt(
    transaction,
    health_result,
    historical_evidence,
    rag_context
):

    return f"""
You are a Retail POS Transaction Investigation Assistant.

Your job is to analyze the supplied evidence and generate a cautious,
evidence-based investigation hypothesis.

IMPORTANT RULES:

1. Do not invent transaction facts.
2. Do not claim that correlation proves causation.
3. Do not describe a hypothesis as a confirmed root cause.
4. Use only the transaction evidence and retrieved POS knowledge supplied below.
5. If evidence is insufficient, explicitly say that the root cause is inconclusive.
6. Financial calculations have already been performed by deterministic tools.
7. Recommend engineering investigation rather than automatically changing code.

TRANSACTION
-----------
Transaction ID: {transaction['transaction_id']}
Store: {transaction['store_id']}
Lane: {transaction['lane_id']}

Gift Card Used: {transaction['gift_card']['used']}
Gift Card Amount: ${transaction['gift_card']['amount']:.2f}
Gift Card Status: {transaction['gift_card']['activation_status']}
Current Gift Card Failure Amount:
${transaction['gift_card']['failure_amount']:.2f}

Expected Cashback:
${health_result['expected_cashback']:.2f}

Actual Cashback:
${health_result['actual_cashback']:.2f}

Cashback Variance:
${health_result['variance']:.2f}

Anomaly:
{health_result['anomaly_type']}


HISTORICAL EVIDENCE
-------------------
{historical_evidence['observation']}

Interpretation:
{historical_evidence['interpretation']}


RETRIEVED POS KNOWLEDGE
-----------------------
{rag_context}


Generate the response using exactly these sections:

OBSERVATION

SUPPORTING EVIDENCE

ROOT-CAUSE HYPOTHESIS

CONFIDENCE
Use only: HIGH, MEDIUM, LOW or INCONCLUSIVE.

RECOMMENDED INVESTIGATION

HUMAN REVIEW
State whether engineering review is required.
"""

Build the prompt for TXN006

In [55]:
investigation_prompt = build_investigation_prompt(
    historical_transactions[2],
    txn006_health,
    historical_evidence,
    rag_context
)

print(investigation_prompt)


You are a Retail POS Transaction Investigation Assistant.

Your job is to analyze the supplied evidence and generate a cautious,
evidence-based investigation hypothesis.

IMPORTANT RULES:

1. Do not invent transaction facts.
2. Do not claim that correlation proves causation.
3. Do not describe a hypothesis as a confirmed root cause.
4. Use only the transaction evidence and retrieved POS knowledge supplied below.
5. If evidence is insufficient, explicitly say that the root cause is inconclusive.
6. Financial calculations have already been performed by deterministic tools.
7. Recommend engineering investigation rather than automatically changing code.

TRANSACTION
-----------
Transaction ID: TXN006
Store: STORE103
Lane: LANE05

Gift Card Used: False
Gift Card Amount: $0.00
Gift Card Status: NOT_APPLICABLE
Current Gift Card Failure Amount:
$0.00

Expected Cashback:
$5.00

Actual Cashback:
$30.00

Cashback Variance:
$25.00

Anomaly:
HIGH_CASHBACK


HISTORICAL EVIDENCE
-------------------


Call Gemini

In [56]:
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")

print("Key exists:", api_key is not None)
print("Key length:", len(api_key.strip()) if api_key else 0)

Key exists: True
Key length: 52


In [57]:
from google import genai
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY").strip()

client = genai.Client(api_key=api_key)

print("Client created successfully")

Client created successfully


In [58]:
client = genai.Client(api_key=api_key)

In [66]:
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")

print("Secret found:", api_key is not None)
print("Length:", len(api_key) if api_key else 0)
print("Contains newline:", "\n" in api_key if api_key else None)

Secret found: True
Length: 52
Contains newline: False


Part 7 — Define the Multi-Agent Workflow

## Part 7: Multi-Agent Workflow

The POS Transaction Anomaly Investigation Assistant uses specialized
agents with clearly separated responsibilities.

Agents:

1. Transaction Agent
   - Validates and summarizes the completed POS transaction.

2. Anomaly Detection Agent
   - Uses deterministic health-check tools to identify NORMAL,
     HIGH_CASHBACK, or NEGATIVE_CASHBACK conditions.

3. Pattern Analysis Agent
   - Searches for monetary correlations within the current transaction.

4. Historical Context Agent
   - Searches previous transactions for cross-transaction correlations.

5. RAG Knowledge Agent
   - Retrieves relevant POS troubleshooting and business guidance.

6. RCA Investigation Agent
   - Uses GenAI to combine evidence and retrieved knowledge to generate
     a probable root-cause hypothesis.

7. Human Review
   - The generated hypothesis is approved or escalated for further
     engineering investigation.

Build our Router

In [67]:
def route_transaction(health_result):

    anomaly_type = health_result["anomaly_type"]

    if health_result["status"] == "NORMAL":
        return "NORMAL_PATH"

    elif anomaly_type == "HIGH_CASHBACK":
        return "HIGH_CASHBACK_PATH"

    elif anomaly_type == "NEGATIVE_CASHBACK":
        return "NEGATIVE_CASHBACK_PATH"

    else:
        return "MANUAL_REVIEW_PATH"

In [68]:
for transaction in transactions:

    health = check_transaction_health(transaction)
    route = route_transaction(health)

    print(
        transaction["transaction_id"],
        "->",
        health["anomaly_type"],
        "->",
        route
    )

TXN001 -> NONE -> NORMAL_PATH
TXN002 -> HIGH_CASHBACK -> HIGH_CASHBACK_PATH
TXN003 -> NEGATIVE_CASHBACK -> NEGATIVE_CASHBACK_PATH


Create the Transaction Agent

In [69]:
def transaction_agent(transaction):

    item_total = round(
        sum(
            item["price"] * item["quantity"]
            for item in transaction["items"]
        ),
        2
    )

    return {
        "transaction_id": transaction["transaction_id"],
        "store_id": transaction["store_id"],
        "lane_id": transaction["lane_id"],

        "item_count": len(transaction["items"]),
        "calculated_item_total": item_total,

        "basket_total": transaction["basket_total"],

        "gift_card_used": transaction["gift_card"]["used"],
        "gift_card_status": transaction["gift_card"]["activation_status"],
        "gift_card_amount": transaction["gift_card"]["amount"],
        "gift_card_failure_amount":
            transaction["gift_card"]["failure_amount"],

        "payment_type": transaction["payment"]["type"],
        "payment_status": transaction["payment"]["status"],

        "expected_cashback": transaction["cashback"]["expected"],
        "actual_cashback": transaction["cashback"]["actual"]
    }

In [70]:
summary = transaction_agent(transactions[1])

print(json.dumps(summary, indent=2))

{
  "transaction_id": "TXN002",
  "store_id": "STORE101",
  "lane_id": "LANE02",
  "item_count": 2,
  "calculated_item_total": 70.0,
  "basket_total": 70.0,
  "gift_card_used": true,
  "gift_card_status": "FAILED",
  "gift_card_amount": 25.0,
  "gift_card_failure_amount": 25.0,
  "payment_type": "CREDIT",
  "payment_status": "SUCCESS",
  "expected_cashback": 5.0,
  "actual_cashback": 30.0
}


Create the Anomaly Detection Agent

In [71]:
def anomaly_detection_agent(transaction):

    health_result = check_transaction_health(transaction)

    route = route_transaction(health_result)

    return {
        "health_result": health_result,
        "route": route
    }

In [72]:
result = anomaly_detection_agent(transactions[1])

print(json.dumps(result, indent=2))

{
  "health_result": {
    "transaction_id": "TXN002",
    "status": "ANOMALY",
    "anomaly_type": "HIGH_CASHBACK",
    "expected_cashback": 5.0,
    "actual_cashback": 30.0,
    "variance": 25.0
  },
  "route": "HIGH_CASHBACK_PATH"
}


Pattern Agent

In [73]:
def pattern_analysis_agent(transaction, health_result):

    result = analyze_amount_correlations(
        transaction,
        health_result
    )

    result["evidence_strength"] = (
        calculate_evidence_strength(result)
    )

    return result

In [74]:
health = check_transaction_health(transactions[1])

pattern_result = pattern_analysis_agent(
    transactions[1],
    health
)

print(json.dumps(pattern_result, indent=2))

{
  "transaction_id": "TXN002",
  "anomaly_type": "HIGH_CASHBACK",
  "investigation_basis": "EXCESS_CASHBACK",
  "suspicious_amount": 25.0,
  "correlations": [
    {
      "type": "EXACT_MATCH",
      "field": "gift_card_amount",
      "value": 25.0
    },
    {
      "type": "EXACT_MATCH",
      "field": "gift_card_failure_amount",
      "value": 25.0
    }
  ],
  "evidence_strength": "STRONG"
}


Historical Agent

In [75]:
def historical_context_agent(transaction, history):

    result = analyze_historical_context(
        transaction,
        history
    )

    evidence = build_historical_evidence(result)

    return {
        "analysis": result,
        "evidence": evidence
    }

In [76]:
history_agent_result = historical_context_agent(
    historical_transactions[2],
    historical_transactions
)

print(json.dumps(history_agent_result, indent=2))

{
  "analysis": {
    "transaction_id": "TXN006",
    "anomaly_type": "HIGH_CASHBACK",
    "suspicious_amount": 25.0,
    "previous_failure_amounts": [
      10.0,
      15.0
    ],
    "accumulated_failure_amount": 25.0,
    "historical_correlation": true
  },
  "evidence": {
    "evidence_type": "CROSS_TRANSACTION_CORRELATION",
    "strength": "STRONG",
    "observation": "The suspicious cashback amount of $25.00 matches the accumulated previous gift-card failure amount of $25.00.",
    "interpretation": "This may indicate that failure-related state persisted across transaction boundaries. Further investigation is required."
  }
}


RAG Agent

In [77]:
def rag_knowledge_agent(query, top_k=2):

    documents = retrieve_pos_knowledge(
        query,
        top_k=top_k
    )

    return {
        "query": query,
        "documents": documents
    }

In [78]:
rag_result = rag_knowledge_agent(
    """
    High cashback may correlate with gift card
    failure state retained across transactions.
    What should be investigated?
    """
)

for doc in rag_result["documents"]:
    print(doc["title"])

Gift Card Failure Handling Guide
POS Cashback Processing Guide


Build a Complete Investigation State

In [79]:
def create_investigation_state(transaction):

    return {
        "transaction": transaction,

        "transaction_summary": None,
        "health_result": None,
        "route": None,

        "pattern_evidence": None,
        "historical_evidence": None,

        "rag_documents": None,

        "rca_result": None,

        "human_decision": None,

        "errors": []
    }

In [80]:
state = create_investigation_state(transactions[1])

print(json.dumps(state, indent=2))

{
  "transaction": {
    "transaction_id": "TXN002",
    "store_id": "STORE101",
    "lane_id": "LANE02",
    "items": [
      {
        "item_id": "ITEM101",
        "price": 40.0,
        "quantity": 1
      },
      {
        "item_id": "ITEM102",
        "price": 30.0,
        "quantity": 1
      }
    ],
    "basket_total": 70.0,
    "promotion": {
      "discount_amount": 5.0
    },
    "gift_card": {
      "used": true,
      "amount": 25.0,
      "activation_status": "FAILED",
      "failure_amount": 25.0
    },
    "payment": {
      "type": "CREDIT",
      "amount": 70.0,
      "status": "SUCCESS"
    },
    "cashback": {
      "expected": 5.0,
      "actual": 30.0
    }
  },
  "transaction_summary": null,
  "health_result": null,
  "route": null,
  "pattern_evidence": null,
  "historical_evidence": null,
  "rag_documents": null,
  "rca_result": null,
  "human_decision": null,
  "errors": []
}


Part 8 — LangGraph Multi-Agent Orchestration# New Section

## Part 8: LangGraph Multi-Agent Orchestration

LangGraph is used to orchestrate the specialized POS investigation agents.

Workflow:

1. Transaction Agent
2. Anomaly Detection Agent
3. Router
4. Pattern Analysis Agent
5. Historical Context Agent
6. RAG Knowledge Agent
7. RCA Investigation Agent
8. Human Review

Normal transactions exit the workflow without invoking deeper
investigation agents.

Abnormal transactions are routed through the investigation workflow.

In [81]:
!pip -q install langgraph

In [82]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Optional, Any

print("LangGraph imported successfully.")

LangGraph imported successfully.


Define the shared state

In [83]:
class POSInvestigationState(TypedDict, total=False):

    transaction: dict

    transaction_summary: dict

    health_result: dict
    route: str

    pattern_evidence: dict
    historical_evidence: dict

    rag_query: str
    rag_documents: list

    rca_result: dict

    human_decision: str

    errors: list

Transaction node

In [84]:
def transaction_node(state):

    transaction = state["transaction"]

    try:
        summary = transaction_agent(transaction)

        return {
            "transaction_summary": summary
        }

    except Exception as e:

        return {
            "errors": [f"Transaction Agent Error: {str(e)}"]
        }

Anomaly node

In [85]:
def anomaly_node(state):

    transaction = state["transaction"]

    try:
        result = anomaly_detection_agent(transaction)

        return {
            "health_result": result["health_result"],
            "route": result["route"]
        }

    except Exception as e:

        return {
            "errors": [f"Anomaly Agent Error: {str(e)}"],
            "route": "MANUAL_REVIEW_PATH"
        }

Router

In [86]:
def anomaly_router(state):

    route = state.get("route")

    if route == "NORMAL_PATH":
        return "normal"

    elif route in [
        "HIGH_CASHBACK_PATH",
        "NEGATIVE_CASHBACK_PATH"
    ]:
        return "investigate"

    else:
        return "manual"

Pattern Analysis node

In [87]:
def pattern_node(state):

    try:
        result = pattern_analysis_agent(
            state["transaction"],
            state["health_result"]
        )

        return {
            "pattern_evidence": result
        }

    except Exception as e:

        return {
            "errors": [f"Pattern Agent Error: {str(e)}"]
        }

Historical Context node

In [88]:
all_history = transactions + historical_transactions

In [89]:
def history_node(state):

    try:
        result = historical_context_agent(
            state["transaction"],
            all_history
        )

        return {
            "historical_evidence": result
        }

    except Exception as e:

        return {
            "errors": [f"Historical Agent Error: {str(e)}"]
        }

RAG node

In [90]:
def rag_node(state):

    try:

        health = state["health_result"]
        pattern = state.get("pattern_evidence", {})
        history = state.get("historical_evidence", {})

        query = f"""
        POS transaction anomaly investigation.

        Anomaly:
        {health.get('anomaly_type')}

        Cashback variance:
        {health.get('variance')}

        Current transaction correlations:
        {pattern.get('correlations', [])}

        Historical evidence:
        {history.get('evidence', {})}

        Retrieve relevant POS guidance for investigating
        this transaction anomaly.
        """

        result = rag_knowledge_agent(
            query,
            top_k=2
        )

        return {
            "rag_query": query,
            "rag_documents": result["documents"]
        }

    except Exception as e:

        return {
            "errors": [f"RAG Agent Error: {str(e)}"]
        }

Temporary RCA Agent

In [91]:
def rca_node(state):

    health = state["health_result"]
    pattern = state.get("pattern_evidence", {})
    history = state.get("historical_evidence", {})

    pattern_matches = pattern.get("correlations", [])

    historical_match = (
        history
        .get("analysis", {})
        .get("historical_correlation", False)
    )

    if historical_match:

        hypothesis = (
            "A cross-transaction monetary correlation was detected. "
            "The abnormal cashback amount matches accumulated "
            "gift-card failure amounts from previous transactions. "
            "Transaction-state handling should be investigated."
        )

        confidence = "HIGH"

    elif pattern_matches:

        fields = [
            match["field"]
            for match in pattern_matches
        ]

        hypothesis = (
            "The cashback anomaly correlates with current "
            "transaction values: "
            + ", ".join(fields)
            + ". Further investigation is required."
        )

        confidence = "MEDIUM"

    else:

        hypothesis = (
            "No sufficiently strong monetary correlation was "
            "identified. The root cause is inconclusive."
        )

        confidence = "INCONCLUSIVE"

    return {
        "rca_result": {
            "anomaly": health["anomaly_type"],
            "hypothesis": hypothesis,
            "confidence": confidence,
            "requires_human_review": True,
            "llm_status": "PENDING_API_AUTHENTICATION"
        }
    }

Human Review node

In [92]:
def human_review_node(state):

    rca = state.get("rca_result", {})

    return {
        "human_decision": "PENDING_REVIEW"
    }

Build the graph

In [93]:
workflow = StateGraph(POSInvestigationState)

workflow.add_node(
    "transaction_agent",
    transaction_node
)

workflow.add_node(
    "anomaly_agent",
    anomaly_node
)

workflow.add_node(
    "pattern_agent",
    pattern_node
)

workflow.add_node(
    "history_agent",
    history_node
)

workflow.add_node(
    "rag_agent",
    rag_node
)

workflow.add_node(
    "rca_agent",
    rca_node
)

workflow.add_node(
    "human_review",
    human_review_node
)

In [94]:
workflow.add_edge(
    START,
    "transaction_agent"
)

workflow.add_edge(
    "transaction_agent",
    "anomaly_agent"
)

In [95]:
workflow.add_conditional_edges(
    "anomaly_agent",
    anomaly_router,
    {
        "normal": END,
        "investigate": "pattern_agent",
        "manual": END
    }
)

In [96]:
workflow.add_edge(
    "pattern_agent",
    "history_agent"
)

workflow.add_edge(
    "history_agent",
    "rag_agent"
)

workflow.add_edge(
    "rag_agent",
    "rca_agent"
)

workflow.add_edge(
    "rca_agent",
    "human_review"
)

workflow.add_edge(
    "human_review",
    END
)

In [97]:
pos_investigation_graph = workflow.compile()

print("POS Investigation Graph compiled successfully.")

POS Investigation Graph compiled successfully.


Test NORMAL transaction

In [98]:
normal_result = pos_investigation_graph.invoke({
    "transaction": transactions[0],
    "errors": []
})

print(json.dumps(normal_result, indent=2))

{
  "transaction": {
    "transaction_id": "TXN001",
    "store_id": "STORE101",
    "lane_id": "LANE01",
    "items": [
      {
        "item_id": "ITEM001",
        "price": 20.0,
        "quantity": 2
      },
      {
        "item_id": "ITEM002",
        "price": 10.0,
        "quantity": 1
      }
    ],
    "basket_total": 50.0,
    "promotion": {
      "discount_amount": 5.0
    },
    "gift_card": {
      "used": false,
      "amount": 0.0,
      "activation_status": "NOT_APPLICABLE",
      "failure_amount": 0.0
    },
    "payment": {
      "type": "DEBIT",
      "amount": 45.0,
      "status": "SUCCESS"
    },
    "cashback": {
      "expected": 5.0,
      "actual": 5.0
    }
  },
  "transaction_summary": {
    "transaction_id": "TXN001",
    "store_id": "STORE101",
    "lane_id": "LANE01",
    "item_count": 2,
    "calculated_item_total": 50.0,
    "basket_total": 50.0,
    "gift_card_used": false,
    "gift_card_status": "NOT_APPLICABLE",
    "gift_card_amount": 0.0,
    "g

Test HIGH cashback

In [99]:
high_result = pos_investigation_graph.invoke({
    "transaction": transactions[1],
    "errors": []
})

print(json.dumps(high_result, indent=2))

{
  "transaction": {
    "transaction_id": "TXN002",
    "store_id": "STORE101",
    "lane_id": "LANE02",
    "items": [
      {
        "item_id": "ITEM101",
        "price": 40.0,
        "quantity": 1
      },
      {
        "item_id": "ITEM102",
        "price": 30.0,
        "quantity": 1
      }
    ],
    "basket_total": 70.0,
    "promotion": {
      "discount_amount": 5.0
    },
    "gift_card": {
      "used": true,
      "amount": 25.0,
      "activation_status": "FAILED",
      "failure_amount": 25.0
    },
    "payment": {
      "type": "CREDIT",
      "amount": 70.0,
      "status": "SUCCESS"
    },
    "cashback": {
      "expected": 5.0,
      "actual": 30.0
    }
  },
  "transaction_summary": {
    "transaction_id": "TXN002",
    "store_id": "STORE101",
    "lane_id": "LANE02",
    "item_count": 2,
    "calculated_item_total": 70.0,
    "basket_total": 70.0,
    "gift_card_used": true,
    "gift_card_status": "FAILED",
    "gift_card_amount": 25.0,
    "gift_card_fail

Test our historical edge case

In [102]:
historical_result = pos_investigation_graph.invoke({
    "transaction": historical_transactions[2],
    "errors": []
})

print(json.dumps(
    historical_result["rca_result"],
    indent=2
))

{
  "anomaly": "HIGH_CASHBACK",
  "hypothesis": "A cross-transaction monetary correlation was detected. The abnormal cashback amount matches accumulated gift-card failure amounts from previous transactions. Transaction-state handling should be investigated.",
  "confidence": "HIGH",
  "requires_human_review": true,
  "llm_status": "PENDING_API_AUTHENTICATION"
}


Part 9 — Parallel Investigation & Observability

## Part 9: Parallel Investigation and Observability

After an anomaly is detected, current-transaction pattern analysis and
historical-context analysis are independent investigation activities.

LangGraph can execute these branches independently before combining
their evidence for RAG and root-cause analysis.

The workflow also records execution logs so that the investigation path
can be observed and audited.

Add simple execution logging

In [106]:
from datetime import datetime

execution_log = []

def log_agent(agent_name, transaction_id, message):

    entry = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "transaction_id": transaction_id,
        "agent": agent_name,
        "message": message
    }

    execution_log.append(entry)

In [107]:
def transaction_node_v2(state):

    transaction = state["transaction"]

    try:
        result = transaction_agent(transaction)

        log_agent(
            "Transaction Agent",
            transaction["transaction_id"],
            "Transaction parsed successfully"
        )

        return {"transaction_summary": result}

    except Exception as e:

        log_agent(
            "Transaction Agent",
            transaction.get("transaction_id", "UNKNOWN"),
            f"ERROR: {str(e)}"
        )

        return {
            "errors": [f"Transaction Agent Error: {str(e)}"]
        }

In [108]:
def anomaly_node_v2(state):

    transaction = state["transaction"]

    try:
        result = anomaly_detection_agent(transaction)

        log_agent(
            "Anomaly Detection Agent",
            transaction["transaction_id"],
            f"Route selected: {result['route']}"
        )

        return {
            "health_result": result["health_result"],
            "route": result["route"]
        }

    except Exception as e:

        log_agent(
            "Anomaly Detection Agent",
            transaction.get("transaction_id", "UNKNOWN"),
            f"ERROR: {str(e)}"
        )

        return {
            "errors": [f"Anomaly Agent Error: {str(e)}"],
            "route": "MANUAL_REVIEW_PATH"
        }

Pattern and History nodes

In [109]:
def pattern_node_v2(state):

    transaction = state["transaction"]

    try:
        result = pattern_analysis_agent(
            transaction,
            state["health_result"]
        )

        log_agent(
            "Pattern Analysis Agent",
            transaction["transaction_id"],
            f"{len(result.get('correlations', []))} correlation(s) found"
        )

        return {"pattern_evidence": result}

    except Exception as e:

        log_agent(
            "Pattern Analysis Agent",
            transaction["transaction_id"],
            f"ERROR: {str(e)}"
        )

        return {
            "pattern_evidence": {},
            "errors": [f"Pattern Agent Error: {str(e)}"]
        }

In [111]:
def history_node_v2(state):

    transaction = state["transaction"]

    try:
        result = historical_context_agent(
            transaction,
            all_history
        )

        log_agent(
            "Historical Context Agent",
            transaction["transaction_id"],
            f"Historical correlation: "
            f"{result.get('analysis', {}).get('historical_correlation', False)}"
        )

        return {"historical_evidence": result}

    except Exception as e:

        log_agent(
            "Historical Context Agent",
            transaction["transaction_id"],
            f"ERROR: {str(e)}"
        )

        return {
            "historical_evidence": {},
            "errors": [f"Historical Agent Error: {str(e)}"]
        }

Create a join node

In [112]:
def evidence_join_node(state):

    transaction_id = state["transaction"]["transaction_id"]

    log_agent(
        "Evidence Aggregator",
        transaction_id,
        "Pattern and historical evidence combined"
    )

    return {}

Build Version 2 LangGraph

In [113]:
workflow_v2 = StateGraph(POSInvestigationState)

workflow_v2.add_node(
    "transaction_agent",
    transaction_node_v2
)

workflow_v2.add_node(
    "anomaly_agent",
    anomaly_node_v2
)

workflow_v2.add_node(
    "pattern_agent",
    pattern_node_v2
)

workflow_v2.add_node(
    "history_agent",
    history_node_v2
)

workflow_v2.add_node(
    "evidence_join",
    evidence_join_node
)

workflow_v2.add_node(
    "rag_agent",
    rag_node
)

workflow_v2.add_node(
    "rca_agent",
    rca_node
)

workflow_v2.add_node(
    "human_review",
    human_review_node
)

In [114]:
workflow_v2.add_edge(
    START,
    "transaction_agent"
)

workflow_v2.add_edge(
    "transaction_agent",
    "anomaly_agent"
)

In [115]:
workflow_v2.add_conditional_edges(
    "anomaly_agent",
    anomaly_router,
    {
        "normal": END,
        "investigate": "pattern_agent",
        "manual": END
    }
)

In [116]:
workflow_v2.add_edge(
    "pattern_agent",
    "history_agent"
)

workflow_v2.add_edge(
    "history_agent",
    "evidence_join"
)

workflow_v2.add_edge(
    "evidence_join",
    "rag_agent"
)

workflow_v2.add_edge(
    "rag_agent",
    "rca_agent"
)

workflow_v2.add_edge(
    "rca_agent",
    "human_review"
)

workflow_v2.add_edge(
    "human_review",
    END
)

In [117]:
pos_investigation_graph_v2 = workflow_v2.compile()

print("Version 2 workflow compiled successfully.")

Version 2 workflow compiled successfully.


One robustness test

In [118]:
invalid_transaction = {
    "transaction_id": "TXN_BAD",
    "store_id": "STORE999",
    "lane_id": "LANE99"
}

In [119]:
execution_log.clear()

bad_result = pos_investigation_graph_v2.invoke({
    "transaction": invalid_transaction,
    "errors": []
})

print(bad_result.get("errors"))

for entry in execution_log:
    print(
        entry["agent"],
        "->",
        entry["message"]
    )

["Anomaly Agent Error: 'cashback'"]
Transaction Agent -> ERROR: 'items'
Anomaly Detection Agent -> ERROR: 'cashback'


Part 10 — Evaluation

## Part 10: Evaluation and Test Results

The prototype is evaluated using synthetic POS transaction scenarios.

The evaluation verifies:

- Transaction anomaly detection accuracy
- Correct workflow routing
- Current-transaction correlation detection
- Historical correlation detection
- RAG retrieval relevance
- Safe handling of inconclusive cases
- Error handling for invalid transaction input

The RCA GenAI evaluation will be performed separately after LLM
authentication is successfully configured.

Create the test scenarios

In [120]:
evaluation_cases = [
    {
        "transaction": transactions[0],
        "expected_anomaly": "NONE",
        "description": "Normal transaction"
    },
    {
        "transaction": transactions[1],
        "expected_anomaly": "HIGH_CASHBACK",
        "description": "High cashback with current gift-card correlation"
    },
    {
        "transaction": transactions[2],
        "expected_anomaly": "NEGATIVE_CASHBACK",
        "description": "Negative cashback with no direct correlation"
    },
    {
        "transaction": historical_transactions[2],
        "expected_anomaly": "HIGH_CASHBACK",
        "description": "High cashback with historical failure correlation"
    }
]

print("Evaluation cases:", len(evaluation_cases))

Evaluation cases: 4


Evaluate anomaly detection

In [121]:
evaluation_results = []

for case in evaluation_cases:

    transaction = case["transaction"]

    health = check_transaction_health(transaction)

    correct = (
        health["anomaly_type"]
        == case["expected_anomaly"]
    )

    evaluation_results.append({
        "transaction_id": transaction["transaction_id"],
        "scenario": case["description"],
        "expected": case["expected_anomaly"],
        "detected": health["anomaly_type"],
        "correct": correct
    })

In [122]:
import pandas as pd

evaluation_df = pd.DataFrame(evaluation_results)

display(evaluation_df)

,transaction_id,scenario,expected,detected,correct
0,TXN001,Normal transaction,NONE,NONE,True
1,TXN002,High cashback with current gift-card correlation,HIGH_CASHBACK,HIGH_CASHBACK,True
2,TXN003,Negative cashback with no direct correlation,NEGATIVE_CASHBACK,NEGATIVE_CASHBACK,True
3,TXN006,High cashback with historical failure correlation,HIGH_CASHBACK,HIGH_CASHBACK,True


Calculate detection accuracy

In [123]:
accuracy = (
    evaluation_df["correct"].sum()
    / len(evaluation_df)
) * 100

print(
    f"Anomaly Detection Accuracy: {accuracy:.2f}%"
)

Anomaly Detection Accuracy: 100.00%


Evaluate investigation evidence

In [124]:
investigation_evaluation = []

for case in evaluation_cases:

    transaction = case["transaction"]
    health = check_transaction_health(transaction)

    if health["status"] == "NORMAL":

        investigation_evaluation.append({
            "transaction_id": transaction["transaction_id"],
            "anomaly": "NONE",
            "current_correlation": False,
            "historical_correlation": False,
            "evidence_result": "NOT_REQUIRED"
        })

        continue

    pattern = pattern_analysis_agent(
        transaction,
        health
    )

    history = historical_context_agent(
        transaction,
        all_history
    )

    current_match = (
        len(pattern.get("correlations", [])) > 0
    )

    historical_match = (
        history
        .get("analysis", {})
        .get("historical_correlation", False)
    )

    if historical_match:
        evidence_result = "HISTORICAL_CORRELATION"

    elif current_match:
        evidence_result = "CURRENT_TRANSACTION_CORRELATION"

    else:
        evidence_result = "INSUFFICIENT_EVIDENCE"

    investigation_evaluation.append({
        "transaction_id": transaction["transaction_id"],
        "anomaly": health["anomaly_type"],
        "current_correlation": current_match,
        "historical_correlation": historical_match,
        "evidence_result": evidence_result
    })

In [125]:
investigation_df = pd.DataFrame(
    investigation_evaluation
)

display(investigation_df)

,transaction_id,anomaly,current_correlation,historical_correlation,evidence_result
0,TXN001,NONE,False,False,NOT_REQUIRED
1,TXN002,HIGH_CASHBACK,True,False,CURRENT_TRANSACTION_CORRELATION
2,TXN003,NEGATIVE_CASHBACK,False,False,INSUFFICIENT_EVIDENCE
3,TXN006,HIGH_CASHBACK,False,True,HISTORICAL_CORRELATION


Test RAG retrieval

In [126]:
rag_test_cases = [
    {
        "query":
        "Gift card failure amount may affect a later transaction",
        "expected_keyword":
        "Transaction State"
    },
    {
        "query":
        "How should negative cashback be investigated?",
        "expected_keyword":
        "Negative Cashback"
    },
    {
        "query":
        "What should happen when investigation evidence is insufficient?",
        "expected_keyword":
        "Escalation"
    }
]

In [127]:
rag_evaluation_results = []

for case in rag_test_cases:

    docs = retrieve_pos_knowledge(
        case["query"],
        top_k=2
    )

    retrieved_titles = [
        doc["title"]
        for doc in docs
    ]

    success = any(
        case["expected_keyword"].lower()
        in title.lower()
        for title in retrieved_titles
    )

    rag_evaluation_results.append({
        "query": case["query"],
        "expected_topic": case["expected_keyword"],
        "retrieved_documents":
            " | ".join(retrieved_titles),
        "relevant_retrieval": success
    })

In [128]:
rag_eval_df = pd.DataFrame(
    rag_evaluation_results
)

display(rag_eval_df)

,query,expected_topic,retrieved_documents,relevant_retrieval
0,Gift card failure amount may affect a later tr...,Transaction State,Gift Card Failure Handling Guide | POS Transac...,True
1,How should negative cashback be investigated?,Negative Cashback,Negative Cashback Investigation Guide | POS Ca...,True
2,What should happen when investigation evidence...,Escalation,POS Anomaly Escalation Guide | Negative Cashba...,True


Create a final prototype summary

In [129]:
print("=" * 65)
print("AI POS TRANSACTION ANOMALY ASSISTANT")
print("PROTOTYPE EVALUATION SUMMARY")
print("=" * 65)

print(
    f"Anomaly test scenarios : "
    f"{len(evaluation_cases)}"
)

print(
    f"Correct classifications: "
    f"{evaluation_df['correct'].sum()}"
)

print(
    f"Detection accuracy     : "
    f"{accuracy:.2f}%"
)

print(
    f"RAG test queries       : "
    f"{len(rag_test_cases)}"
)

print(
    f"Relevant RAG retrievals: "
    f"{rag_eval_df['relevant_retrieval'].sum()}"
    f"/{len(rag_eval_df)}"
)

print()
print(
    "LLM RCA Status         : "
    "Pending Gemini authentication"
)

AI POS TRANSACTION ANOMALY ASSISTANT
PROTOTYPE EVALUATION SUMMARY
Anomaly test scenarios : 4
Correct classifications: 4
Detection accuracy     : 100.00%
RAG test queries       : 3
Relevant RAG retrievals: 3/3

LLM RCA Status         : Pending Gemini authentication


Part 11 — Local GenAI RCA Agent

## Part 11: GenAI Root-Cause Investigation Agent

The RCA Agent uses a language model to combine:

- Transaction anomaly information
- Current transaction correlation evidence
- Historical transaction evidence
- Retrieved RAG knowledge

The agent generates an evidence-grounded root-cause hypothesis,
recommended investigation steps, and human-review recommendation.

A local Hugging Face model is used for the prototype so that the
workflow does not depend on an external LLM API.

Install Transformers

In [133]:
!pip -q install transformers accelerate sentencepiece

In [134]:
from transformers import pipeline
import torch

print("GPU available:", torch.cuda.is_available())

GPU available: False


Load the model

In [135]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)

print("FLAN-T5 model loaded successfully.")
print("Device:", device)

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

FLAN-T5 model loaded successfully.
Device: cpu


In [136]:
test_prompt = "Explain in one sentence what a retail POS transaction anomaly is."

inputs = tokenizer(
    test_prompt,
    return_tensors="pt"
).to(device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=80
    )

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(answer)

A retail POS transaction anomaly occurs when a customer is unable to enter a transaction.


In [137]:
def genai_rca_agent(state):

    context = build_rca_context(state)

    prompt = f"""
You are a retail POS transaction investigation assistant.

Analyze only the supplied evidence.

Do not invent facts.
Do not treat correlation as confirmed causation.
If evidence is insufficient, state that the root cause is inconclusive.

Evidence:

{context}

Provide a concise investigation containing:

Observation:
Root-cause hypothesis:
Supporting evidence:
Recommended investigation:
Human review:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=220,
            do_sample=False
        )

    generated_text = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return {
        "rca_result": {
            "generated_analysis": generated_text,
            "requires_human_review": True,
            "model": "google/flan-t5-base"
        }
    }

In [139]:
def build_rca_context(state):

    health = state.get("health_result", {})
    pattern = state.get("pattern_evidence", {})
    history = state.get("historical_evidence", {})
    rag_docs = state.get("rag_documents", [])

    pattern_matches = pattern.get("correlations", [])

    historical_info = history.get("evidence", {})

    rag_text = "\n".join([
        f"{doc['title']}: {doc['content']}"
        for doc in rag_docs
    ])

    return f"""
Anomaly Type:
{health.get('anomaly_type')}

Expected Cashback:
{health.get('expected_cashback')}

Actual Cashback:
{health.get('actual_cashback')}

Cashback Variance:
{health.get('variance')}

Current Transaction Correlations:
{pattern_matches}

Historical Evidence:
{historical_info}

Relevant POS Knowledge:
{rag_text}
"""

In [140]:
print("build_rca_context available:", callable(build_rca_context))

build_rca_context available: True


In [141]:
print("historical_result available:", "historical_result" in globals())

historical_result available: True


In [142]:
historical_result = pos_investigation_graph.invoke({
    "transaction": historical_transactions[2],
    "errors": []
})

In [143]:
historical_result = pos_investigation_graph_v2.invoke({
    "transaction": historical_transactions[2],
    "errors": []
})

In [144]:
print("Tokenizer:", "tokenizer" in globals())
print("Model:", "model" in globals())
print("Device:", "device" in globals())

Tokenizer: True
Model: True
Device: True


In [145]:
def genai_rca_agent(state):

    context = build_rca_context(state)

    prompt = f"""
You are a retail POS transaction investigation assistant.

Analyze only the supplied evidence.

Do not invent facts.
Do not treat correlation as confirmed causation.
If evidence is insufficient, state that the root cause is inconclusive.

Evidence:

{context}

Provide a concise investigation containing:

Observation:
Root-cause hypothesis:
Supporting evidence:
Recommended investigation:
Human review:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=220,
            do_sample=False
        )

    generated_text = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return {
        "rca_result": {
            "generated_analysis": generated_text,
            "requires_human_review": True,
            "model": "google/flan-t5-base"
        }
    }

In [146]:
llm_rca_result = genai_rca_agent(
    historical_result
)

print(
    llm_rca_result["rca_result"]["generated_analysis"]
)

Anomaly is a monetary anomaly. Anomaly is a monetary anomaly. Anomaly is a monetary anomaly.


In [147]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: False


In [148]:
del model
del tokenizer

import gc
gc.collect()

print("Old model cleared.")

Old model cleared.


In [149]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32
)

model.eval()

print("Qwen model loaded successfully.")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen model loaded successfully.


In [150]:
messages = [
    {
        "role": "system",
        "content": "You are a retail POS transaction investigation assistant."
    },
    {
        "role": "user",
        "content": """
A POS transaction expected $5 cashback but produced $30 cashback.

The excess cashback is therefore $25.

Two earlier transactions on the same lane had gift-card failure
amounts of $10 and $15.

The accumulated previous failure amount is $25.

What should an engineer investigate?

Do not claim that correlation proves the root cause.
"""
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=180,
        do_sample=False
    )

new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

answer = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
)

print(answer)

An engineer investigating this issue would likely focus on several key areas to determine if there's a root cause for the higher-than-expected cashback:

1. **Transaction Details:**
   - **Cashback Calculation:** Ensure that the calculation of the cashback is accurate, considering any discounts or additional charges applied during the transaction.
   - **Gift Card Failure:** Verify that the gift cards were successfully processed and whether they resulted in any cashback or other fees.

2. **POS System Logs:**
   - Review the logs from the POS system to see if there was any unusual behavior or error messages related to the transaction.
   - Check for any discrepancies between the actual transaction amounts and those displayed by the system.

3. **Customer Service Interaction:**
   - Analyze customer service interactions with the POS system to ensure that there were no issues with the payment process or receipt of the cashback.



Part 11 — Final Qwen RCA Agent

In [151]:
def genai_rca_agent(state):

    context = build_rca_context(state)

    messages = [
        {
            "role": "system",
            "content": """
You are a Retail POS Transaction Anomaly Investigation Assistant.

You analyze evidence produced by deterministic investigation tools.

Rules:
- Use only the supplied evidence.
- Never invent transaction facts.
- Explicitly identify important monetary correlations.
- Correlation is evidence, not proof of causation.
- Never describe a hypothesis as a confirmed root cause.
- If evidence is insufficient, say INCONCLUSIVE.
- Recommend engineering investigation and human review.
"""
        },
        {
            "role": "user",
            "content": f"""
Investigate this POS cashback anomaly.

EVIDENCE
========
{context}

Produce the result using exactly these headings:

OBSERVATION:
State the anomaly and important monetary values.

SUPPORTING EVIDENCE:
Identify current or historical correlations.
Explicitly mention matching amounts when present.

ROOT-CAUSE HYPOTHESIS:
Provide only a probable hypothesis supported by the evidence.
Do not claim it is confirmed.

RECOMMENDED INVESTIGATION:
State what an engineer should inspect in the POS processing,
transaction state, failure handling, or logs.

HUMAN REVIEW:
State whether human engineering review is required.
"""
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    )

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False,
            repetition_penalty=1.1
        )

    new_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    generated_text = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    )

    return {
        "rca_result": {
            "generated_analysis": generated_text,
            "requires_human_review": True,
            "model": "Qwen2.5-0.5B-Instruct"
        }
    }

In [152]:
llm_rca_result = genai_rca_agent(
    historical_result
)

print(
    llm_rca_result["rca_result"]["generated_analysis"]
)

**OBSERVATION:** The observed cashback variance of 25% indicates that there might be a discrepancy between the expected cashback and the actual cashback amount. This could suggest that the transaction has been affected by some form of error or issue related to the cashback calculation process.

**SUPPORTING EVIDENCE:** 
1. **Cross-Transaction Correlation**: The observation about the cumulative previous gift-card failure amount of $25.00 aligns with the observed cashback variance of 25%. This suggests that the transaction's cashback amount is influenced by past financial events, which could be indicative of fraud or manipulation.

2. **POS Transaction State Management Guide**: The presence of a "STATE" associated with the transaction does not directly correlate with the observed cashback variance. However, if the transaction is part of a larger sequence of transactions where the state management guide is applied consistently, it could provide additional context.

**ROOT-CAUSE HYPOTHESIS